# 실습 1: Attention과 Transformer 직접 돌려보기 (라이브러리로 한 번에 실행)

이 실습은 PyTorch의 어텐션·트랜스포머 **라이브러리 모듈**(`nn.MultiheadAttention`, `nn.TransformerEncoderLayer`)을 사용해, 짧은 아이템 시퀀스에 셀프 어텐션과 트랜스포머를 **처음부터 끝까지** 한 번에 돌려보는 오버뷰(Overview) 실습입니다. 코드 해설 없이 **무엇이 어떤 모양으로 흘러가는지**를 눈으로 확인하는 데 집중합니다.

**개념 복기 및 이론 점검**
- 셀프 어텐션은 시퀀스 안의 각 위치가 **다른 모든 위치를 직접 바라본다**고 합니다. 그렇다면 출력 어텐션 가중치 행렬의 한 행은 무엇을 의미할까요?
- 다음 아이템을 예측할 때 **미래 아이템을 보면 안 되는** 이유는 무엇일까요? (causal masking)
- 트랜스포머는 RNN과 달리 순서 정보를 내장하지 않습니다. 그렇다면 순서는 어떻게 알려줄까요?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/5주차/lab_01_attention.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

## 1. 환경 준비: 라이브러리 임포트

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## 2. 입력 시퀀스 준비: 아이템 임베딩 + 위치 임베딩
한 사용자가 순서대로 본 아이템 시퀀스를 임베딩으로 바꾸고, 순서를 알려주는 위치 임베딩을 더합니다.

In [ ]:
# 한 사용자가 순서대로 소비한 아이템 시퀀스 (toy 예시)
num_items = 20          # 전체 아이템 종류 수
d_model = 16            # 임베딩 차원
seq = torch.tensor([[3, 7, 1, 9, 4]])   # (batch=1, seq_len=5)
seq_len = seq.size(1)

item_emb = nn.Embedding(num_items, d_model)
pos_emb = nn.Embedding(seq_len, d_model)

positions = torch.arange(seq_len).unsqueeze(0)   # (1, seq_len)
x = item_emb(seq) + pos_emb(positions)           # (1, seq_len, d_model)

print("입력 임베딩 shape:", x.shape)

## 3. 셀프 어텐션 (nn.MultiheadAttention)
각 위치가 모든 위치를 바라보며 관계(어텐션 가중치)를 계산합니다.

In [ ]:
mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=2, batch_first=True)

attn_out, attn_w = mha(x, x, x, need_weights=True)

print("어텐션 출력 shape:", attn_out.shape)   # (1, seq_len, d_model)
print("어텐션 가중치 shape:", attn_w.shape)    # (1, seq_len, seq_len)
print("\n어텐션 가중치(각 행의 합 = 1):\n", attn_w[0].detach().round(decimals=2))

## 4. Causal Masking: 미래 가리기
다음 아이템을 예측하려면 아직 일어나지 않은 미래를 참조하면 안 됩니다. 상삼각을 −∞로 막습니다.

In [ ]:
causal_mask = torch.triu(
    torch.full((seq_len, seq_len), float('-inf')), diagonal=1)

masked_out, masked_w = mha(x, x, x, attn_mask=causal_mask, need_weights=True)

print("causal mask:\n", causal_mask)
print("\n마스킹 후 어텐션 가중치(상삼각=0):\n", masked_w[0].detach().round(decimals=2))

## 5. 어텐션 가중치 시각화
마스킹 전후의 어텐션 가중치를 히트맵으로 비교합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, w, title in [(axes[0], attn_w, "Self-Attention"),
                     (axes[1], masked_w, "Causal Self-Attention")]:
    im = ax.imshow(w[0].detach(), cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xlabel("Key position (past items)")
    ax.set_ylabel("Query position (current step)")
    ax.set_xticks(range(seq_len)); ax.set_yticks(range(seq_len))
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## 6. Transformer 인코더 (nn.TransformerEncoderLayer)
어텐션 + 잔차연결 + LayerNorm + 피드포워드를 묶은 표준 블록을 쌓습니다.

In [ ]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=d_model, nhead=2, dim_feedforward=64, batch_first=True)
encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)

out = encoder(x, mask=causal_mask)
print("Transformer 인코더 출력 shape:", out.shape)   # (1, seq_len, d_model)

## 7. 다음 아이템 예측 헤드
각 시점의 출력으로 다음에 올 아이템 점수를 만듭니다.

In [ ]:
head = nn.Linear(d_model, num_items)
logits = head(out)                       # (1, seq_len, num_items)
next_item = logits[:, -1].argmax(dim=-1)  # 마지막 시점의 예측

print("각 시점의 다음-아이템 점수 shape:", logits.shape)
print("마지막 시점이 예측한 다음 아이템 ID:", next_item.item(),
      "(학습 전이라 무작위에 가깝습니다)")

## 8. ✅ 학습 결과 정리
- 아이템 시퀀스를 임베딩 + 위치 임베딩으로 바꾸고, `nn.MultiheadAttention`으로 셀프 어텐션을 계산했습니다.
- causal mask로 미래를 가리면 어텐션 가중치의 상삼각이 0이 되는 것을 히트맵으로 확인했습니다.
- `nn.TransformerEncoderLayer`를 쌓아 인코더를 만들고, 예측 헤드로 다음 아이템 점수를 산출했습니다.

🎯 **핵심 결론:** 어텐션과 트랜스포머는 라이브러리 모듈 몇 개로 조립됩니다. 다음 lab에서는 **이 코드를 한 줄씩 뜯어** 각 모듈이 왜 그렇게 동작하는지 원리를 짚어봅니다.